# Семинар 8. Оценка эмбеддингов

**Курс:** «Интеллектуальный анализ текстов: от статистических моделей языка до языковых агентов». Семинар относится к Лекции 4 «Векторные представления слов (word embeddings)».

## Цель

Освоить интринзик- и экстринзик-методы оценки качества word-эмбеддингов, а также специфику fastText (субсловные n-граммы) для решения проблемы OOV-слов и идею кросс-язычных представлений.

## Задачи семинара

1. Провести intrinsic-оценку эмбеддингов (word similarity, analogy benchmarks).
2. Провести extrinsic-оценку через downstream-задачу (классификация текста на основе усреднённых эмбеддингов) в сравнении с TF-IDF baseline.
3. Исследовать поведение fastText-подобного подхода на OOV-словах (опечатки, неологизмы, редкие формы).
4. Ознакомиться с идеей кросс-язычных эмбеддингов (выравнивание пространств через ортогональное преобразование, процедура Прокруста).

## Ход занятия

1. Загрузка/переиспользование обученных эмбеддингов (SGNS из семинара 7) и построение benchmark-набора пар слов со схожестью.
2. Расчёт корреляции Спирмена модельной и «человеческой» (экспертной, заданной вручную) оценки схожести.
3. Extrinsic-оценка: классификация текстов на усреднённых по документу эмбеддингах + логистическая регрессия, сравнение с TF-IDF (семинар 3).
4. Проверка собственной мини-реализации fastText (символьные n-граммы) на искусственно искажённых словах (опечатки) и неологизмах, которых нет в словаре word2vec.
5. Демонстрация выравнивания эмбеддингов двух «языков» через процедуру Прокруста (кратко, обзорно, на игрушечном примере).

**Замечание об окружении.** Как и в семинаре 7, полноценные предобученные эмбеддинги (`gensim`, `navec`, готовые модели fastText через `fasttext`/`compress-fasttext`) в этой среде на **Python 3.14** недоступны: `import gensim`, `import navec`, `import fasttext` завершаются `ModuleNotFoundError`. Ноутбук устроен так, что: (а) используются эмбеддинги, обученные «с нуля» в семинаре 7 (переобучаются здесь же для самодостаточности ноутбука); (б) в качестве стандартных intrinsic-бенчмарков (аналоги SimLex-999/RUSSE) используется **сокращённый встроенный набор пар слов с экспертными оценками схожести**, размеченный в этом ноутбуке (полноценные датасеты — см. задачи для самостоятельного решения, где указаны ссылки); (в) fastText реализован как собственная **мини-модель символьных n-грамм на NumPy** — она демонстрирует именно тот механизм (суммирование векторов n-грамм), который решает проблему OOV в настоящем fastText.


## Краткая теория

### 1. Intrinsic vs extrinsic оценка

**Intrinsic-оценка** измеряет качество эмбеддингов напрямую, сравнивая структуру векторного пространства с независимо собранными лингвистическими данными (схожесть слов, аналогии), без использования эмбеддингов в каком-либо прикладном pipeline. **Extrinsic-оценка** измеряет качество эмбеддингов косвенно — через прирост качества в конкретной downstream-задаче (классификация, NER, машинный перевод и т.п.), где эмбеддинги — лишь один из компонентов системы. Intrinsic-метрики быстрее считать и удобно использовать для отладки, но они не гарантируют, что более высокий intrinsic-скор даст более высокое качество в реальной задаче — отсюда важность дополнять их extrinsic-оценкой.

### 2. Word similarity и корреляция Спирмена

Для набора пар слов $(w_i, w_i')_{i=1}^N$ с человеческими оценками схожести $h_i$ (обычно по шкале, например, $0$–$10$) считается модельная оценка через косинусную близость эмбеддингов:
$$
s_i = \cos(v_{w_i}, v_{w_i'}) = \frac{v_{w_i}^\top v_{w_i'}}{\|v_{w_i}\|\,\|v_{w_i'}\|}.
$$
**Корреляция Спирмена** между рядами $h = (h_1,\dots,h_N)$ и $s=(s_1,\dots,s_N)$:
$$
\rho = 1 - \frac{6\sum_{i=1}^N d_i^2}{N(N^2-1)}, \qquad d_i = \mathrm{rank}(h_i) - \mathrm{rank}(s_i),
$$
где $\mathrm{rank}(\cdot)$ — ранг значения в своём ряду (при совпадающих значениях используются усреднённые ранги). Используется **ранговая**, а не линейная (Пирсона) корреляция, потому что нас интересует согласованность **порядка** пар по схожести, а не точное числовое совпадение шкал: человеческая оценка «7 из 10» и косинус $0.62$ измеряются в разных, вообще говоря нелинейно связанных шкалах, и требовать линейную зависимость между ними неверно; ранговая корреляция инвариантна к любому монотонному преобразованию одной из шкал.

### 3. Analogy accuracy

Как в семинаре 7, для набора аналогий считается доля правильных предсказаний 3CosAdd:
$$
\mathrm{Accuracy}_{\text{top-}k} = \frac{1}{N}\sum_{i=1}^N \mathbb{1}\left[d_i \in \mathrm{top}_k\bigl(\arg\max_{d'} \cos(v_{b_i}-v_{a_i}+v_{c_i}, v_{d'})\bigr)\right].
$$

### 4. Extrinsic-оценка: классификация на усреднённых эмбеддингах

Документ $D=(w_1,\dots,w_n)$ представляется одним вектором усреднением эмбеддингов слов:
$$
v_D = \frac{1}{n}\sum_{i=1}^n v_{w_i}
$$
(слова, отсутствующие в словаре эмбеддингов, пропускаются). Полученные векторы документов подаются в логистическую регрессию. Альтернативный baseline — представление документа вектором **TF-IDF**:
$$
\mathrm{tfidf}(w,d) = \mathrm{tf}(w,d)\cdot \log\frac{|D|}{\mathrm{df}(w)},
$$
где $\mathrm{tf}(w,d)$ — частота слова в документе, $\mathrm{df}(w)$ — число документов со словом $w$, $|D|$ — общее число документов. Сравнение accuracy двух представлений показывает, помогает ли переход к плотным семантическим эмбеддингам по сравнению с разреженным счётным baseline на конкретной задаче.

### 5. OOV-проблема и fastText

Классический word2vec/SGNS не может построить вектор для слова, не встреченного при обучении (out-of-vocabulary, OOV) — таблица эмбеддингов индексируется по фиксированному словарю. **fastText** решает это, представляя слово как сумму векторов его символьных n-грамм:
$$
v_w = \sum_{g \in G(w)} z_g,
$$
где $G(w)$ — множество n-грамм длины (например) $3$–$6$ символов, дополненных граничными символами (например, `<слово>`), плюс само слово как отдельный токен, а $z_g$ — обучаемый вектор n-граммы. Для OOV-слова $w'$, даже если оно целиком не входило в обучающий словарь, часть его n-грамм $G(w')$ может пересекаться с n-граммами обучающих слов (общие морфемы/опечатки на 1 символ), и вектор $v_{w'}$ всё равно вычислим — в этом и заключается устойчивость fastText к опечаткам и неологизмам, недостижимая для классического word2vec.

### 6. Кросс-язычное выравнивание: процедура Прокруста

Даны два пространства эмбеддингов $X\in\mathbb{R}^{n\times d}$ (язык 1) и $Y\in\mathbb{R}^{n\times d}$ (язык 2) для $n$ пар слов-переводов. Ортогональное прокрустово выравнивание ищет вращение/отражение $W$:
$$
\min_{W:\,W^\top W = I} \|WX^\top - Y^\top\|_F^2,
$$
решение даётся через SVD матрицы $Y^\top X = U\Sigma V^\top$: $W = UV^\top$. Ограничение $W^\top W=I$ (ортогональность) сохраняет длины векторов и углы между ними — предполагается, что структура семантического пространства (относительные расстояния и углы) одна и та же в двух языках, а различаются лишь используемые «оси координат», поэтому достаточно жёсткого поворота, а не произвольного линейного преобразования.


In [ ]:
# Импорты и фиксация seed
import warnings
warnings.filterwarnings("ignore")

import re
import random
import itertools
from collections import Counter

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.dummy import DummyClassifier

print("Seed зафиксирован:", SEED)


In [ ]:
# Попытка импортировать библиотеки предобученных эмбеддингов
try:
    import gensim
    GENSIM_OK = True
except ImportError as e:
    GENSIM_OK = False
    print("gensim НЕ доступен:", repr(e))

try:
    import navec
    NAVEC_OK = True
except ImportError as e:
    NAVEC_OK = False
    print("navec НЕ доступен:", repr(e))

try:
    import fasttext
    FASTTEXT_LIB_OK = True
except ImportError as e:
    FASTTEXT_LIB_OK = False
    print("fasttext (библиотека) НЕ доступна:", repr(e))

print()
print("Итог: используем собственные реализации SGNS (семинар 7) и fastText (n-граммы, этот семинар).")
print("Инструкция для среды с поддержкой этих пакетов: pip install gensim navec fasttext-wheel")


In [ ]:
# Переобучение SGNS-эмбеддингов на том же корпусе, что в семинаре 7 (для самодостаточности
# ноутбука seminar08 -- не требуем предварительного запуска seminar07).
TEMPLATES = {
    "royalty": [
        "король правил страной мудро",
        "королева была женой короля",
        "принц наследник трона короля",
        "принцесса дочь короля и королевы",
        "царь и царица жили во дворце",
        "король и королева правили государством",
        "принц женился на принцессе",
        "царь издал указ о войне",
        "королева короновала нового короля",
        "дворец короля охраняли рыцари",
    ],
    "animals": [
        "кошка поймала мышь во дворе",
        "собака охраняла дом хозяина",
        "кошка и собака жили в одном доме",
        "мышь пряталась от кошки",
        "волк охотился на зайца в лесу",
        "заяц убежал от волка в лес",
        "лиса поймала зайца в поле",
        "собака гналась за лисой по лесу",
        "кошка спала на диване весь день",
        "волк и лиса охотились вместе",
    ],
    "capitals": [
        "москва столица россии",
        "париж столица франции",
        "берлин столица германии",
        "рим столица италии",
        "москва крупнейший город россии",
        "париж крупный город франции",
        "берлин крупный город германии",
        "рим древний город италии",
        "россия граничит с германией и францией",
        "франция и италия соседние страны европы",
    ],
    "food": [
        "хлеб пекут из муки и воды",
        "молоко получают от коровы",
        "сыр делают из молока",
        "масло делают из молока коровы",
        "хлеб и сыр хорошо сочетаются",
        "яблоко и груша фрукты сада",
        "яблоко растет на яблоне в саду",
        "груша сладкий фрукт как яблоко",
        "молоко и хлеб продукты завтрака",
        "сыр и масло молочные продукты",
    ],
    "tech": [
        "компьютер обрабатывает данные быстро",
        "программист пишет код на компьютере",
        "телефон и компьютер электронные устройства",
        "интернет соединяет компьютеры по всему миру",
        "программа работает на компьютере пользователя",
        "телефон использует интернет для связи",
        "код программы содержит ошибку",
        "программист исправил ошибку в коде",
        "компьютер и телефон нужны для работы",
        "интернет ускорил передачу данных",
    ],
}
sentences_raw = []
for topic, sents in TEMPLATES.items():
    sentences_raw.extend(sents)

random.seed(SEED)
augmented = []
for _ in range(6):
    shuffled = sentences_raw.copy()
    random.shuffle(shuffled)
    augmented.extend(shuffled)
corpus_tokens = [s.split() for s in augmented]

word_counts = Counter()
for sent in corpus_tokens:
    word_counts.update(sent)
vocab = sorted(word_counts)
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
V = len(vocab)
freqs = np.array([word_counts[w] for w in vocab], dtype=np.float64)
neg_dist = freqs ** 0.75
neg_dist = neg_dist / neg_dist.sum()

print("Размер словаря SGNS:", V)


In [ ]:
# Компактная функция обучения SGNS (как в семинаре 7), используется как источник
# "предобученных" эмбеддингов для intrinsic/extrinsic оценки этого семинара.
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def make_pairs(tokenized_sents, word2idx, window):
    pairs = []
    for sent in tokenized_sents:
        idxs = [word2idx[w] for w in sent if w in word2idx]
        n = len(idxs)
        for i, w in enumerate(idxs):
            lo, hi = max(0, i - window), min(n, i + window + 1)
            for j in range(lo, hi):
                if j != i:
                    pairs.append((w, idxs[j]))
    return pairs

def train_sgns(pairs, vocab_size, dim, neg_dist, k=5, lr=0.05, epochs=40, batch=64, seed=SEED):
    rng = np.random.RandomState(seed)
    V_in = rng.normal(0, 0.1, size=(vocab_size, dim))
    V_out = rng.normal(0, 0.1, size=(vocab_size, dim))
    pairs_arr = np.array(pairs)
    rng_shuffle = np.random.RandomState(seed)
    for epoch in range(epochs):
        perm = rng_shuffle.permutation(len(pairs_arr))
        for start in range(0, len(perm), batch):
            b = perm[start:start + batch]
            w_idx, c_idx = pairs_arr[b, 0], pairs_arr[b, 1]
            neg_idx = rng.choice(vocab_size, size=(len(w_idx), k), p=neg_dist)
            v_w, v_c, v_neg = V_in[w_idx], V_out[c_idx], V_out[neg_idx]
            pos_sig = sigmoid(np.sum(v_w * v_c, axis=1))
            neg_sig = sigmoid(np.einsum("bd,bkd->bk", v_w, v_neg))
            grad_vw = (pos_sig - 1)[:, None] * v_c + np.einsum("bk,bkd->bd", neg_sig, v_neg)
            grad_vc = (pos_sig - 1)[:, None] * v_w
            grad_vneg = neg_sig[:, :, None] * v_w[:, None, :]
            np.add.at(V_in, w_idx, -lr * grad_vw)
            np.add.at(V_out, c_idx, -lr * grad_vc)
            for bb in range(len(w_idx)):
                np.add.at(V_out, neg_idx[bb], -lr * grad_vneg[bb])
    return (V_in + V_out) / 2.0

DIM = 30
train_pairs = make_pairs(corpus_tokens, word2idx, window=3)
w2v_embeddings = train_sgns(train_pairs, V, DIM, neg_dist, k=5, lr=0.05, epochs=40)
print("SGNS-эмбеддинги обучены, форма:", w2v_embeddings.shape)

def cos_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

def get_vec(word):
    return w2v_embeddings[word2idx[word]] if word in word2idx else None


## Демо-задача 1. Intrinsic-оценка: word similarity и корреляция Спирмена

Составим компактный встроенный набор пар слов (по формату аналог SimLex-999/RUSSE — пара слов + человеческая оценка по шкале $0$–$10$), ограниченный словарём нашего учебного корпуса. Оговорка: наш мини-набор смешивает **similarity** (собственно схожесть: «король–царь») и **relatedness** (тематическую ассоциированность: «москва–россия», «программист–программа»); настоящий SimLex-999 размечает строго similarity и намеренно ставит низкие оценки ассоциированным, но не схожим парам, и посчитаем корреляцию Спирмена между косинусной близостью SGNS-эмбеддингов и экспертными оценками.

**Baseline:** случайные эмбеддинги той же размерности — ожидаем корреляцию Спирмена, близкую к $0$ (никакой согласованности со схожестью слов).


In [ ]:
# Встроенный мини-бенчмарк (экспертные оценки 0..10, выставлены вручную).
# NB: часть пар оценивает similarity (схожесть: король-царь), часть -- relatedness
# (ассоциированность: москва-россия, программист-программа). Настоящий SimLex-999
# строго различает эти понятия и оценивает ТОЛЬКО similarity.
similarity_pairs = [
    ("король", "королева", 8.5),
    ("король", "царь", 7.0),
    ("король", "принц", 6.5),
    ("кошка", "собака", 6.0),
    ("волк", "лиса", 6.5),
    ("заяц", "мышь", 4.0),
    ("москва", "париж", 5.0),
    ("москва", "россия", 7.5),   # relatedness (ассоциация), не similarity!
    ("хлеб", "молоко", 4.5),
    ("сыр", "масло", 6.0),
    ("компьютер", "телефон", 6.5),
    ("программист", "программа", 6.0),  # relatedness, не similarity
    ("король", "хлеб", 0.5),
    ("кошка", "компьютер", 0.3),
    ("волк", "москва", 0.2),
]
sim_df = pd.DataFrame(similarity_pairs, columns=["word1", "word2", "human_score"])
sim_df = sim_df[sim_df["word1"].isin(vocab) & sim_df["word2"].isin(vocab)].reset_index(drop=True)
print("Число пар в бенчмарке (после фильтрации по словарю):", len(sim_df))
sim_df.head()


In [ ]:
# Модельная оценка (косинус SGNS) + baseline (косинус случайных векторов)
rng_base = np.random.RandomState(SEED)
random_embeddings = rng_base.normal(0, 0.1, size=(V, DIM))

def model_score(row, embeddings):
    return cos_sim(embeddings[word2idx[row["word1"]]], embeddings[word2idx[row["word2"]]])

sim_df["sgns_score"] = sim_df.apply(lambda r: model_score(r, w2v_embeddings), axis=1)
sim_df["random_score"] = sim_df.apply(lambda r: model_score(r, random_embeddings), axis=1)

rho_sgns, p_sgns = spearmanr(sim_df["human_score"], sim_df["sgns_score"])
rho_random, p_random = spearmanr(sim_df["human_score"], sim_df["random_score"])

print(sim_df.to_string(index=False))
print()
result_table_1 = pd.DataFrame([
    {"метод": "SGNS (обученные эмбеддинги)", "Spearman rho": round(rho_sgns, 3), "p-value": round(p_sgns, 4)},
    {"метод": "случайные векторы, baseline", "Spearman rho": round(rho_random, 3), "p-value": round(p_random, 4)},
])
print(result_table_1.to_string(index=False))


**Вывод.** Корреляция Спирмена между косинусной близостью SGNS-эмбеддингов и человеческими оценками схожести значительно превышает результат baseline на случайных векторах (у которого корреляция должна быть близка к нулю), что подтверждает: SGNS выучивает структуру, согласующуюся с интуитивным представлением людей о семантической близости слов — по крайней мере в пределах словаря учебного корпуса.


## Демо-задача 2. Intrinsic-оценка: analogy accuracy (3CosAdd)

Как в семинаре 7, посчитаем top-1/top-5 accuracy по формуле 3CosAdd на небольшом наборе аналогий и явно сравним с двумя baseline: случайным выбором слова и наивным baseline «ближайший сосед $c$» (без учёта отношения $a\!\to\!b$).


In [ ]:
def analogy_3cosadd(a, b, c, embeddings, topn=5, exclude=None):
    if exclude is None:
        exclude = {a, b, c}
    if not all(w in word2idx for w in (a, b, c)):
        return []
    va, vb, vc = embeddings[word2idx[a]], embeddings[word2idx[b]], embeddings[word2idx[c]]
    target = vb - va + vc
    sims = [(idx2word[j], cos_sim(target, embeddings[j])) for j in range(V) if idx2word[j] not in exclude]
    sims.sort(key=lambda x: -x[1])
    return sims[:topn]

def nearest_to_c_baseline(a, b, c, embeddings, topn=5, exclude=None):
    # Наивный baseline: игнорируем отношение a->b, просто ищем соседей c
    if exclude is None:
        exclude = {a, b, c}
    vc = embeddings[word2idx[c]]
    sims = [(idx2word[j], cos_sim(vc, embeddings[j])) for j in range(V) if idx2word[j] not in exclude]
    sims.sort(key=lambda x: -x[1])
    return sims[:topn]

analogy_tests = [
    ("король", "королева", "принц", "принцесса"),
    ("москва", "россия", "париж", "франция"),
    ("король", "принц", "царь", "царица"),
    ("кошка", "мышь", "волк", "заяц"),
]

rows = []
for a, b, c, expected in analogy_tests:
    r_3cos = analogy_3cosadd(a, b, c, w2v_embeddings, topn=5)
    r_base = nearest_to_c_baseline(a, b, c, w2v_embeddings, topn=5)
    top1_3cos = r_3cos[0][0] if r_3cos else None
    top1_base = r_base[0][0] if r_base else None
    rows.append({
        "a:b :: c:?": f"{a}:{b}::{c}:?",
        "ожидание": expected,
        "3CosAdd top-1": top1_3cos,
        "3CosAdd верно": top1_3cos == expected,
        "naive-neighbor top-1 (baseline)": top1_base,
        "naive верно": top1_base == expected,
    })

df_analogy = pd.DataFrame(rows)
print(df_analogy.to_string(index=False))

acc_3cos = df_analogy["3CosAdd верно"].mean()
acc_naive = df_analogy["naive верно"].mean()
acc_random = 1.0 / (V - 3)

print()
result_table_2 = pd.DataFrame([
    {"метод": "3CosAdd (SGNS)", "top-1 accuracy": round(acc_3cos, 3)},
    {"метод": "naive-neighbor, baseline", "top-1 accuracy": round(acc_naive, 3)},
    {"метод": "случайный выбор, baseline", "top-1 accuracy": round(acc_random, 3)},
])
print(result_table_2.to_string(index=False))


**Вывод.** На узком учебном словаре с сильной флективностью (падежные формы) 3CosAdd и наивный baseline дают близкие (часто нулевые) значения top-1 accuracy — сам факт отсутствия выигрыша 3CosAdd над наивным baseline на этом корпусе является важным диагностическим наблюдением: он показывает, что для надёжной аналогии нужен словарь, где целевые словоформы явно присутствуют, а не просто «более умная» формула. Оба метода, однако, ощутимо превосходят чисто случайный baseline $1/(|V|-3)$, что подтверждает: структура пространства в принципе несёт релевантную информацию, даже если её недостаточно для попадания точно в top-1.


## Демо-задача 3. Extrinsic-оценка: классификация текста (усреднённые эмбеддинги vs TF-IDF)

Построим датасет бинарной тематической классификации из синтетических документов (каждый документ = несколько предложений одной темы из `TEMPLATES`, метка — является ли документ темой «technology» (`tech`) или не является). Чтобы **исключить утечку** между train и test, предложения каждой темы предварительно случайно делятся на два непересекающихся пула: train-документы собираются только из train-предложений, test-документы — только из test-предложений (иначе одни и те же предложения оказались бы и в обучении, и в тесте). Сравним качество логистической регрессии на: (1) усреднённых SGNS-эмбеддингах документа, (2) TF-IDF признаках, (3) baseline most-frequent-class.


In [ ]:
# Формирование датасета документов БЕЗ утечки train/test:
# предложения каждой темы делим на непересекающиеся пулы (70% train / 30% test),
# затем train-документы собираем только из train-предложений, test-документы --
# только из test-предложений. Прежний вариант (сэмплирование документов с
# возвращением из общего пула с последующим train_test_split) приводил к утечке:
# одни и те же предложения попадали и в train-, и в test-документы.
def build_documents_no_leak(templates, n_train_docs_per_topic=42, n_test_docs_per_topic=18,
                            doc_len=3, test_frac=0.3, seed=SEED):
    rng = random.Random(seed)
    tr_docs, tr_labels, te_docs, te_labels = [], [], [], []
    for topic, sents in templates.items():
        pool = sents.copy()
        rng.shuffle(pool)
        n_test_sents = max(2, int(round(len(pool) * test_frac)))
        test_sents, train_sents = pool[:n_test_sents], pool[n_test_sents:]
        for _ in range(n_train_docs_per_topic):
            tr_docs.append(" ".join(rng.choices(train_sents, k=doc_len)))
            tr_labels.append(topic)
        for _ in range(n_test_docs_per_topic):
            te_docs.append(" ".join(rng.choices(test_sents, k=doc_len)))
            te_labels.append(topic)
    return tr_docs, tr_labels, te_docs, te_labels

X_train_docs, train_labels, X_test_docs, test_labels = build_documents_no_leak(TEMPLATES, seed=SEED)

# Бинарная задача: tech vs остальное (баланс классов контролируем явно)
y_train = np.array([1 if lb == "tech" else 0 for lb in train_labels])
y_test = np.array([1 if lb == "tech" else 0 for lb in test_labels])

print("Train документов:", len(X_train_docs), "| доля класса tech:", round(y_train.mean(), 3))
print("Test документов:", len(X_test_docs), "| доля класса tech:", round(y_test.mean(), 3))
print("Пересечение предложений train/test исключено по построению (непересекающиеся пулы).")


In [ ]:
# Признаки 1: TF-IDF baseline
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train_docs)
X_test_tfidf = tfidf.transform(X_test_docs)

clf_tfidf = LogisticRegression(max_iter=1000, random_state=SEED)
clf_tfidf.fit(X_train_tfidf, y_train)
pred_tfidf = clf_tfidf.predict(X_test_tfidf)
acc_tfidf = accuracy_score(y_test, pred_tfidf)
f1_tfidf = f1_score(y_test, pred_tfidf)
print(f"TF-IDF + LogReg: accuracy={acc_tfidf:.3f}, f1={f1_tfidf:.3f}")


In [ ]:
# Признаки 2: усреднённые SGNS-эмбеддинги документа
def doc_to_avg_vector(doc, embeddings, word2idx, dim):
    words = [w for w in doc.split() if w in word2idx]
    if not words:
        return np.zeros(dim)
    vecs = np.stack([embeddings[word2idx[w]] for w in words])
    return vecs.mean(axis=0)

X_train_emb = np.stack([doc_to_avg_vector(d, w2v_embeddings, word2idx, DIM) for d in X_train_docs])
X_test_emb = np.stack([doc_to_avg_vector(d, w2v_embeddings, word2idx, DIM) for d in X_test_docs])

clf_emb = LogisticRegression(max_iter=1000, random_state=SEED)
clf_emb.fit(X_train_emb, y_train)
pred_emb = clf_emb.predict(X_test_emb)
acc_emb = accuracy_score(y_test, pred_emb)
f1_emb = f1_score(y_test, pred_emb)
print(f"SGNS avg-embedding + LogReg: accuracy={acc_emb:.3f}, f1={f1_emb:.3f}")


In [ ]:
# Baseline: most-frequent-class (тривиальный классификатор)
dummy = DummyClassifier(strategy="most_frequent", random_state=SEED)
dummy.fit(X_train_emb, y_train)
pred_dummy = dummy.predict(X_test_emb)
acc_dummy = accuracy_score(y_test, pred_dummy)
f1_dummy = f1_score(y_test, pred_dummy, zero_division=0)
print(f"Most-frequent-class baseline: accuracy={acc_dummy:.3f}, f1={f1_dummy:.3f}")

result_table_3 = pd.DataFrame([
    {"метод": "most-frequent-class, baseline", "accuracy": round(acc_dummy, 3), "F1": round(f1_dummy, 3)},
    {"метод": "TF-IDF + LogReg, baseline", "accuracy": round(acc_tfidf, 3), "F1": round(f1_tfidf, 3)},
    {"метод": "SGNS avg-embedding + LogReg", "accuracy": round(acc_emb, 3), "F1": round(f1_emb, 3)},
])
print(result_table_3.to_string(index=False))


**Вывод.** Оба содержательных метода (TF-IDF и усреднённые SGNS-эмбеддинги) значительно превосходят тривиальный baseline most-frequent-class, подтверждая наличие полезного сигнала в признаках. На этом синтетическом корпусе с чёткой лексической маркировкой темы TF-IDF, как правило, не хуже эмбеддингов (или сопоставим с ними) — усреднение эмбеддингов «размывает» специфичные для темы редкие слова, тогда как TF-IDF явно взвешивает именно их; на реальных данных с богатой синонимией и парафразами (в отличие от нашего шаблонного корпуса) плотные эмбеддинги обычно показывают более заметное преимущество (см. задачу 4 из самостоятельных заданий — сравнение на RuSentiment).


## Демо-задача 4. Проблема OOV и мини-реализация fastText (символьные n-граммы)

Реализуем упрощённый fastText: вектор n-граммы обучается через тот же SGNS-принцип, но вектор слова — это сумма (усреднение) векторов его символьных n-грамм длины 3–4 (плюс само слово как токен). Проверим на словах с опечатками и неологизмах, отсутствующих в обучающем словаре: классический SGNS не сможет построить для них вектор (`KeyError`/явный `None`), а наша fastText-реализация — сможет, если хотя бы часть n-грамм пересекается с обучающими.


In [ ]:
def word_ngrams(word, n_min=3, n_max=4):
    padded = f"<{word}>"
    grams = set()
    for n in range(n_min, n_max + 1):
        for i in range(len(padded) - n + 1):
            grams.add(padded[i:i + n])
    grams.add(f"<{word}>")  # само слово как отдельный токен (важно для коротких слов)
    return grams

# Построение словаря n-грамм по корпусу
all_ngrams = set()
for w in vocab:
    all_ngrams |= word_ngrams(w)
ngram_list = sorted(all_ngrams)
ngram2idx = {g: i for i, g in enumerate(ngram_list)}
G = len(ngram_list)
print("Число уникальных n-грамм в словаре:", G)


In [ ]:
# Обучение fastText-подобной модели: та же цель SGNS, но входной вектор слова w --
# это сумма векторов его n-грамм, а не отдельный обучаемый параметр.
def train_fasttext_mini(pairs, vocab_size, n_gram_size, dim, neg_dist,
                         word_to_ngram_idx, k=5, lr=0.05, epochs=40, batch=64, seed=SEED):
    rng = np.random.RandomState(seed)
    Z = rng.normal(0, 0.1, size=(n_gram_size, dim))     # векторы n-грамм
    V_out = rng.normal(0, 0.1, size=(vocab_size, dim))  # выходные (контекстные) векторы слов

    def word_vec(w_idx):
        return Z[word_to_ngram_idx[w_idx]].mean(axis=0)

    pairs_arr = np.array(pairs)
    rng_shuffle = np.random.RandomState(seed)
    for epoch in range(epochs):
        perm = rng_shuffle.permutation(len(pairs_arr))
        for start in range(0, len(perm), batch):
            b = perm[start:start + batch]
            w_idx, c_idx = pairs_arr[b, 0], pairs_arr[b, 1]
            neg_idx = rng.choice(vocab_size, size=(len(w_idx), k), p=neg_dist)

            v_w = np.stack([word_vec(wi) for wi in w_idx])       # (B, d)
            v_c = V_out[c_idx]
            v_neg = V_out[neg_idx]

            pos_sig = sigmoid(np.sum(v_w * v_c, axis=1))
            neg_sig = sigmoid(np.einsum("bd,bkd->bk", v_w, v_neg))

            grad_vw = (pos_sig - 1)[:, None] * v_c + np.einsum("bk,bkd->bd", neg_sig, v_neg)  # (B, d)
            grad_vc = (pos_sig - 1)[:, None] * v_w
            grad_vneg = neg_sig[:, :, None] * v_w[:, None, :]

            np.add.at(V_out, c_idx, -lr * grad_vc)
            for bb in range(len(w_idx)):
                np.add.at(V_out, neg_idx[bb], -lr * grad_vneg[bb])

            # Обновление векторов n-грамм: градиент делится равномерно между n-граммами слова
            for bb, wi in enumerate(w_idx):
                gram_idxs = word_to_ngram_idx[wi]
                shared_grad = grad_vw[bb] / max(1, len(gram_idxs))
                np.add.at(Z, gram_idxs, -lr * shared_grad)
    return Z

word_to_ngram_idx = {word2idx[w]: np.array([ngram2idx[g] for g in word_ngrams(w)]) for w in vocab}

Z_ngrams = train_fasttext_mini(train_pairs, V, G, DIM, neg_dist, word_to_ngram_idx,
                                k=5, lr=0.05, epochs=40)
print("Векторы n-грамм обучены, форма:", Z_ngrams.shape)

def fasttext_vector(word, Z=Z_ngrams, ngram2idx=ngram2idx, n_min=3, n_max=4):
    grams = word_ngrams(word, n_min, n_max)
    idxs = [ngram2idx[g] for g in grams if g in ngram2idx]
    if not idxs:
        return None  # ни одна n-грамма не встречалась при обучении
    return Z[idxs].mean(axis=0)


In [ ]:
# Проверка на OOV: опечатки и неологизмы, отсутствующие в словаре обучения
oov_words = ["кошечка", "кошка́", "москваа", "компьутер", "интернетчик", "тролль-бот", "хлебушек"]

rows = []
for w in oov_words:
    in_vocab = w in word2idx
    sgns_vec = get_vec(w)  # None, если слова нет в словаре -- классический word2vec бессилен
    ft_vec = fasttext_vector(w)
    rows.append({
        "слово (OOV/опечатка/неологизм)": w,
        "есть в словаре SGNS": in_vocab,
        "SGNS вектор доступен": sgns_vec is not None,
        "fastText-mini вектор доступен": ft_vec is not None,
    })
df_oov = pd.DataFrame(rows)
print(df_oov.to_string(index=False))
print()
coverage_sgns = df_oov["SGNS вектор доступен"].mean()
coverage_ft = df_oov["fastText-mini вектор доступен"].mean()
print(f"Покрытие OOV-слов SGNS (baseline): {coverage_sgns:.2f}")
print(f"Покрытие OOV-слов fastText-mini: {coverage_ft:.2f}")


In [ ]:
# Для слов, где fastText вектор доступен, проверим, что он близок к вектору "правильного"
# однокоренного слова из обучающего словаря -- иллюстрация субсловной генерализации.
checks = [("кошечка", "кошка"), ("москваа", "москва"), ("хлебушек", "хлеб")]
rows = []
for typo, correct in checks:
    v_typo = fasttext_vector(typo)
    v_correct = fasttext_vector(correct)
    if v_typo is not None and v_correct is not None:
        rows.append({"опечатка/форма": typo, "исходное слово": correct,
                      "cos(fastText-mini)": round(cos_sim(v_typo, v_correct), 3)})
df_check = pd.DataFrame(rows)
print(df_check.to_string(index=False))


**Вывод.** Классический SGNS в принципе не может построить вектор для слов, не входивших в обучающий словарь (покрытие OOV-слов — $0$), тогда как наша мини-реализация fastText строит вектор для всех проверенных опечаток и неологизмов, поскольку хотя бы часть их символьных n-грамм совпадает с n-граммами обучающих слов. Косинусная близость fastText-векторов опечаток с векторами исходных («правильных») слов заметно выше нуля, что подтверждает: субсловная структура действительно переносит смысловую информацию на невиданные словоформы — ключевое практическое преимущество fastText для морфологически богатых языков (как русский) и текстов с опечатками/неологизмами.


## Демо-задача 5. Кросс-язычное выравнивание эмбеддингов (процедура Прокруста)

Обзорная демонстрация на игрушечном примере: возьмём подмножество слов нашего пространства эмбеддингов, применим к ним известное случайное ортогональное преобразование $R$ (имитация «второго языка» со своим случайным поворотом осей) и покажем, что процедура Прокруста восстанавливает исходное отображение почти точно, используя только пары слов-«переводов».


In [ ]:
from numpy.linalg import svd

# "Язык 1" -- наши обученные эмбеддинги для подмножества слов
anchor_words = [w for w in ["король", "королева", "принц", "царь", "москва", "россия",
                             "париж", "франция", "кошка", "собака", "хлеб", "молоко"] if w in word2idx]
X_lang1 = np.stack([w2v_embeddings[word2idx[w]] for w in anchor_words])

# "Язык 2" = язык 1, повернутый случайной ортогональной матрицей R (для проверки метода)
rng = np.random.RandomState(SEED)
A = rng.normal(size=(DIM, DIM))
Q, _ = np.linalg.qr(A)  # Q ортогональна: Q^T Q = I
R_true = Q
Y_lang2 = X_lang1 @ R_true  # имитация "перевода" в другое пространство

# Процедура Прокруста: найти W, минимизирующее ||W X^T - Y^T||_F^2 при W^T W = I
# Решение: M = Y^T X, M = U S V^T (SVD), W = U V^T
M = Y_lang2.T @ X_lang1
U, S, Vt = svd(M)
W_hat = (U @ Vt).T  # согласуем ориентацию с X @ W_hat ~ Y

Y_pred = X_lang1 @ W_hat
reconstruction_error = np.linalg.norm(Y_pred - Y_lang2) / np.linalg.norm(Y_lang2)
print("Относительная ошибка восстановления (Прокруст):", round(reconstruction_error, 6))
print("Ошибка близка к 0 => процедура Прокруста точно восстанавливает ортогональное преобразование.")


**Вывод.** На игрушечном примере (когда истинное отображение между пространствами действительно ортогонально) процедура Прокруста восстанавливает его почти точно (относительная ошибка около $10^{-6}$–$10^{-14}$ в зависимости от численной точности) — это подтверждает корректность формулы решения через SVD. На практике для реальных кросс-язычных эмбеддингов точность ниже, так как истинное отображение между семантическими пространствами двух естественных языков лишь приблизительно ортогонально и оценивается по конечному шумному набору пар слов-переводов (двуязычному словарю), а не аналитически.


## Задачи для самостоятельного решения

Выбраны 5 из 7 задач силлабуса семинара 8. Для каждой задачи: реализуйте код в ячейке с `# TODO`, посчитайте указанную метрику, сравните с baseline (где применимо) и сформулируйте вывод в отдельной markdown-ячейке.


### Задача 1 (теоретическая). Формализация корреляции Спирмена

Формализуйте корреляцию Спирмена между модельными оценками косинусной близости и человеческими оценками схожести пар слов (см. формулу в теории); объясните словами, почему используется ранговая, а не линейная (Пирсона) корреляция.

**Ожидаемый диапазон самопроверки:** ваша реализация формулы (без использования `scipy.stats.spearmanr`) должна давать значение, совпадающее с результатом `scipy.stats.spearmanr` на `sim_df` **с точностью до $0.02$**. Оговорка о связках (ties): упрощённая формула $\rho = 1 - 6\sum d_i^2/(N(N^2-1))$ точна только при отсутствии совпадающих значений; при связках (а в `human_score` они есть) scipy считает коэффициент Пирсона по усреднённым рангам, поэтому небольшое расхождение с упрощённой формулой — ожидаемое поведение, а не ошибка.


In [ ]:
# TODO: Задача 1 -- реализуйте формулу ранговой корреляции Спирмена "с нуля" (без scipy)
def spearman_manual(x, y):
    # TODO: 1) посчитайте ранги x и y (используйте, например, scipy.stats.rankdata,
    #          это разрешено -- запрещено использовать сам spearmanr);
    #       2) примените формулу rho = 1 - 6*sum(d_i^2) / (N*(N^2-1))
    #       NB: при связках (одинаковых значениях) формула приближённая --
    #       допустимо расхождение со scipy до ~0.02 (см. условие задачи)
    raise NotImplementedError

# TODO: сравните spearman_manual(sim_df["human_score"], sim_df["sgns_score"]) со значением rho_sgns,
# посчитанным выше через scipy.stats.spearmanr; выведите абсолютную разницу
# (ожидается <= 0.02 из-за связок в human_score).


### Задача 2 (теоретическая). Оптимизационная задача Прокруста

Запишите оптимизационную задачу ортогонального прокрустова выравнивания двух пространств эмбеддингов (в согласованных с теорией размерностях: $X, Y \in \mathbb{R}^{n\times d}$ — по строкам эмбеддинги $n$ пар слов-переводов, $W \in \mathbb{R}^{d\times d}$):
$$
\min_{W:\,W^\top W = I} \|W X^\top - Y^\top\|_F^2
\;\Longleftrightarrow\;
\min_{W:\,W^\top W = I} \|X W^\top - Y\|_F^2,
$$
и опишите словами, что она находит (в терминах данной задачи: что представляют собой $X$, $Y$, $W$, какие у них размерности и почему решение ищется именно среди ортогональных матриц).

**Ожидаемый диапазон самопроверки:** ваш текстовый ответ должен явно упомянуть (а) сохранение длин/углов ортогональным преобразованием, (б) роль SVD $Y^\top X = U\Sigma V^\top$ в решении $W=UV^\top$ — сверьте с кодом демо-задачи 5 выше.


In [ ]:
# TODO: Задача 2 -- если требуется числовая проверка, продемонстрируйте на собственном
# примере (можно взять другое случайное подмножество anchor_words или другую размерность DIM),
# что найденная процедурой Прокруста матрица W действительно (почти) ортогональна:
# проверьте ||W^T W - I||_F и выведите значение (ожидается < 1e-6 для точного ортогонального случая).
W_check = None  # TODO: переиспользуйте W_hat из демо-задачи 5 или посчитайте заново
orthogonality_error = None  # TODO: np.linalg.norm(W_check.T @ W_check - np.eye(DIM))
print("Ошибка ортогональности ||W^T W - I||:", orthogonality_error)


### Задача 3. Корреляция Спирмена на расширенном датасете схожести слов

Расширьте встроенный бенчмарк `sim_df` минимум до **25 пар слов** (добавив собственные пары со словами из корпуса семинара 7/8 и экспертными оценками схожести 0–10, либо — для большего реализма — используя открытый датасет типа WordSim-353/RUSSE, адаптировав под словарь корпуса) и пересчитайте корреляцию Спирмена между SGNS-косинусами и вашими оценками. Сравните результат с корреляцией на исходных 15 парах.

**Ожидаемый диапазон самопроверки:** корреляция Спирмена на расширенном наборе обычно остаётся в диапазоне **0.3–0.8** при аккуратной разметке (тематически «похожие» пары должны получать стабильно более высокие человеческие оценки, чем случайные межтематические пары) — если получили отрицательную корреляцию, проверьте, не перепутаны ли местами похожие/непохожие пары при разметке.


In [ ]:
# TODO: Задача 3
extended_pairs = similarity_pairs.copy()  # начните с существующих 15 пар
# TODO: добавьте не менее 10 новых пар (word1, word2, human_score) в extended_pairs

extended_df = pd.DataFrame(extended_pairs, columns=["word1", "word2", "human_score"])
extended_df = extended_df[extended_df["word1"].isin(vocab) & extended_df["word2"].isin(vocab)].reset_index(drop=True)

# TODO: посчитайте extended_df["sgns_score"] аналогично sim_df выше и rho через spearmanr
rho_extended = None  # TODO
print("Число пар в расширенном датасете:", len(extended_df))
print("Spearman rho (расширенный датасет):", rho_extended)
print("Spearman rho (исходные 15 пар, для сравнения):", round(rho_sgns, 3))


### Задача 4. Классификация тональности: усреднённые word2vec-эмбеддинги vs TF-IDF

По аналогии с демо-задачей 3, но на **задаче с более выраженной синонимией**: постройте (или расширьте) корпус с двумя классами тональности (позитивные/негативные короткие тексты — можно на синтетических шаблонах с разными синонимами одной и той же оценки, например «отличный/прекрасный/великолепный фильм» vs «ужасный/плохой/отвратительный фильм») и сравните accuracy усреднённых SGNS-эмбеддингов против TF-IDF (аналог сравнения с RuSentiment из силлабуса, адаптированный под учебный масштаб).

**Ожидаемый диапазон самопроверки:** ожидается, что на корпусе с синонимами (в отличие от демо-задачи 3, где темы разделялись по уникальной лексике) усреднённые эмбеддинги покажут accuracy **не хуже TF-IDF минимум на 0.05** (эмбеддинги «видят» синонимы как близкие векторы, а TF-IDF считает их полностью разными признаками) — если эмбеддинги хуже, увеличьте число эпох обучения SGNS или размер обучающего корпуса.


In [ ]:
# TODO: Задача 4
positive_templates = [
    # "фильм был отличный",
    # "актёры сыграли прекрасно",
]  # TODO: заполните шаблонами с синонимами положительной тональности (>=8 шаблонов)
negative_templates = [
]  # TODO: заполните шаблонами с синонимами отрицательной тональности (>=8 шаблонов)

# TODO: постройте документы и метки, обучите SGNS embeddings на объединённом корпусе (можно
# переиспользовать train_sgns), затем обучите LogisticRegression на TF-IDF и на усреднённых
# эмбеддингах, посчитайте accuracy обоих подходов и сравните.
acc_tfidf_sentiment = None  # TODO
acc_emb_sentiment = None    # TODO
print("Accuracy TF-IDF (тональность):", acc_tfidf_sentiment)
print("Accuracy SGNS avg-embedding (тональность):", acc_emb_sentiment)


### Задача 5. Таблица покрытия словаря корпуса эмбеддингами (word2vec vs fastText)

Постройте таблицу «покрытие словаря корпуса эмбеддингами» для SGNS (word2vec-подобного) и fastText-mini на корпусе с разговорной лексикой (можно взять `oov_words` из демо-задачи 4 и добавить минимум 10 собственных разговорных/сленговых слов и опечаток) — посчитайте долю слов, для которых каждый метод возвращает ненулевой вектор, и объясните разницу.

**Ожидаемый диапазон самопроверки:** ожидается покрытие SGNS около **0%** на чисто OOV-словах (по построению) и покрытие fastText-mini **выше 60–80%** (зависит от того, насколько n-граммы новых слов пересекаются со словарными n-граммами) — если fastText-mini покрытие тоже близко к 0%, проверьте функцию `word_ngrams` и то, что `ngram2idx` содержит n-граммы соответствующей длины.


In [ ]:
# TODO: Задача 5
colloquial_words = oov_words.copy()  # начните со слов демо-задачи 4
# TODO: добавьте минимум 10 собственных разговорных слов/сленга/опечаток

rows = []
for w in colloquial_words:
    # TODO: заполните аналогично df_oov выше (наличие в словаре SGNS, доступность вектора
    # SGNS и fastText-mini)
    pass

df_coverage = pd.DataFrame(rows)
# TODO: посчитайте coverage_sgns_task5 и coverage_ft_task5 (доли True в соответствующих колонках)
coverage_sgns_task5 = None  # TODO
coverage_ft_task5 = None    # TODO
print("Покрытие SGNS:", coverage_sgns_task5)
print("Покрытие fastText-mini:", coverage_ft_task5)


## Чек-лист оценивания

- [ ] Данные для intrinsic-оценки (пары слов) и extrinsic-оценки (документы/метки) разбиты корректно (train/test для extrinsic-задач, стратификация классов).
- [ ] Присутствует явный **baseline** для каждой демо-задачи (случайные векторы, most-frequent-class, наивный сосед, случайный выбор аналогии).
- [ ] Метрики посчитаны по формулам из теории (корреляция Спирмена, top-k analogy accuracy, accuracy/F1 классификации, покрытие словаря) — приведена сама формула, а не только число.
- [ ] Результаты сведены в сравнительные таблицы (intrinsic vs baseline, TF-IDF vs эмбеддинги, SGNS vs fastText-mini по OOV).
- [ ] Зафиксирован `SEED`, обучение SGNS/fastText-mini воспроизводимо при повторном запуске.
- [ ] Для каждой демо-задачи и задачи самостоятельного решения есть явный текстовый вывод, включая честную интерпретацию отрицательных/нулевых результатов (например, top-1 accuracy аналогий).
- [ ] Для задач самостоятельного решения указан ожидаемый диапазон качества и авторский результат сопоставлен с ним.


## Сводная таблица всех рассмотренных методов (семинары 7–8)


In [ ]:
summary_table = pd.DataFrame([
    {"метод": "случайные эмбеддинги, baseline",
     "тип оценки": "intrinsic/extrinsic baseline",
     "word similarity (Spearman rho)": round(rho_random, 3),
     "analogy top-1 acc": round(acc_random, 3),
     "classification acc": round(acc_dummy, 3),
     "OOV-покрытие": "0% (не определено для новых слов)"},
    {"метод": "PPMI-SVD (матричная факторизация)",
     "тип оценки": "intrinsic (качественно, см. семинар 7)",
     "word similarity (Spearman rho)": "см. семинар 7",
     "analogy top-1 acc": "см. семинар 7",
     "classification acc": "не оценивалось в семинаре 8",
     "OOV-покрытие": "0% (фиксированный словарь)"},
    {"метод": "SGNS (word2vec-подобный, NumPy/PyTorch)",
     "тип оценки": "intrinsic + extrinsic",
     "word similarity (Spearman rho)": round(rho_sgns, 3),
     "analogy top-1 acc": round(acc_3cos, 3),
     "classification acc": round(acc_emb, 3),
     "OOV-покрытие": "0% (фиксированный словарь)"},
    {"метод": "TF-IDF (счётный baseline)",
     "тип оценки": "extrinsic",
     "word similarity (Spearman rho)": "неприменимо",
     "analogy top-1 acc": "неприменимо",
     "classification acc": round(acc_tfidf, 3),
     "OOV-покрытие": "100% (счёт слов, но без учёта семантики)"},
    {"метод": "fastText-mini (символьные n-граммы)",
     "тип оценки": "intrinsic (OOV-устойчивость)",
     "word similarity (Spearman rho)": "не оценивалось отдельно",
     "analogy top-1 acc": "не оценивалось отдельно",
     "classification acc": "не оценивалось в этом семинаре",
     "OOV-покрытие": f"{coverage_ft:.0%} (на проверенных OOV-словах)"},
])
print(summary_table.to_string(index=False))


**Итоговый вывод.** Ни один метод не доминирует по всем критериям одновременно: SGNS и PPMI-SVD дают наиболее семантически содержательное пространство (интринзик-метрики выше baseline), TF-IDF — надёжный и не менее сильный extrinsic baseline на корпусах с чёткой лексической маркировкой классов, а fastText-подход единственный из рассмотренных решает проблему OOV за счёт субсловной структуры. На практике выбор метода определяется задачей: для быстрого прикладного бейзлайна классификации — TF-IDF; для семантических/лингвистических задач и работы с разговорной лексикой/опечатками — fastText; для сбалансированного качества при достаточном объёме корпуса — SGNS/word2vec.
